<H1> Feature Engineering </H1>

This notebook transforms raw creature data into engineered features for HP prediction.

**Input:** `data/dnd5e_monsters_from_json.csv`  
**Output:** `helper_files/engineered_features.parquet`

# Imports and Configs

## Python Imports

In [81]:
import json
import numpy as np
import os
import pandas as pd
import sys

from pathlib import Path


In [82]:
# %pip install pyarrow
# %pip install fastparquet

## Set Paths Dynamically

In [83]:
# Detect execution context and set paths dynamically

# Get the current working directory
cwd = Path.cwd()

# Check if we're in the notebooks directory or project root
if cwd.name == 'notebooks':
    # Running from notebooks directory (in Jupyter)
    DATA_DIR = '../data'
    PICKLED_MODELS_DIR = '../pickled_models'
    MONSTER_BUILDER_DIR = '../monster-builder-v2'
    HELPERS_DIR = './helper_files'
    IO_DIR = './notebooks_io'
    IN_NB_DIR = False
else:
    # Running from project root (via run_three_tier_model.py)
    DATA_DIR = './data'
    PICKLED_MODELS_DIR = './pickled_models'
    MONSTER_BUILDER_DIR = './monster-builder-v2'
    HELPERS_DIR = './notebooks/helper_files'
    IO_DIR = './notebooks/notebooks_io'
    IN_NB_DIR = False

print(f"📁 Execution context detected:")
print(f"   Current directory: {cwd}")
print(f"   Data directory: {DATA_DIR}")
print(f"   Models directory: {PICKLED_MODELS_DIR}")


📁 Execution context detected:
   Current directory: /workspaces/matrix_v0
   Data directory: ./data
   Models directory: ./pickled_models


## Add helper_files to path

In [84]:

# Add helper_files to path
# sys.path.insert(0, '.')
if IN_NB_DIR is True:
    from helper_files import (
        # Config
        CONDITIONS, PHASE2_FEATURES, PHASE2_PENALTIES, get_cr_tier, get_phase3_features,
        # Parsers
        parse_cr, parse_hp, parse_ac, parse_speed, SIZE_ORDINAL_MAP,
        count_proficiencies, has_sense, parse_sense_range, parse_passive_perception,
        count_abilities, parse_legendary_actions, parse_attack_bonus, parse_save_dc,
        parse_dpr_from_json, parse_charge_bonus_attack, parse_legendary_actions_dpr,
        parse_legendary_conditions, extract_spellcaster_level,
        has_advantage_condition, has_disadvantage_condition, has_attackers_advantage,
        get_resistance_multiplier, get_immunity_multiplier, extract_family,
        # Baselines
        get_baseline_hp, get_baseline_ac, get_baseline_attack, get_baseline_dpr,
        get_baseline_dc, get_baseline_size_ordinal, get_baseline_speed_ground,
        get_fly_speed_baseline, get_darkvision_baseline,
    )
    # lazy 5e baselines
    # adjust_hp_baseline, parse_bonus, interp1d
    from helper_files import (
        parse_hp_avg, parse_bonus, adjust_hp_baseline
    )

    print("Imports successful")
else:
    from notebooks.helper_files import (
        # Config
        CONDITIONS, PHASE2_FEATURES, PHASE2_PENALTIES, get_cr_tier, get_phase3_features,
        # Parsers
        parse_cr, parse_hp, parse_ac, parse_speed, SIZE_ORDINAL_MAP,
        count_proficiencies, has_sense, parse_sense_range, parse_passive_perception,
        count_abilities, parse_legendary_actions, parse_attack_bonus, parse_save_dc,
        parse_dpr_from_json, parse_charge_bonus_attack, parse_legendary_actions_dpr,
        parse_legendary_conditions, extract_spellcaster_level,
        has_advantage_condition, has_disadvantage_condition, has_attackers_advantage,
        get_resistance_multiplier, get_immunity_multiplier, extract_family,
        # Baselines
        get_baseline_hp, get_baseline_ac, get_baseline_attack, get_baseline_dpr,
        get_baseline_dc, get_baseline_size_ordinal, get_baseline_speed_ground,
        get_fly_speed_baseline, get_darkvision_baseline,
    )
    # lazy 5e baselines
    from notebooks.helper_files import (
        parse_hp_avg, adjust_hp_baseline, parse_bonus
    )

    print("Imports successful")

Imports successful


# Load Raw Data

In [85]:
# Load main creature data
df = pd.read_csv(f'{DATA_DIR}/dnd5e_monsters_from_json.csv')
print(f"Loaded {len(df)} monsters")
print(f"Columns: {list(df.columns)[:10]}...")


# Filter to only 2014 (original 5e) monsters - exclude 2024 additions
df = df[df['Source_Name'] == 'Free Basic Rules (2014)'].copy()
print(f"📊 Loaded {len(df)} monsters (2014 5e only, excluding 2024 additions)")

# Load Lazy 5e baseline stats
lazy_5e = pd.read_csv(f'{DATA_DIR}/lazy_5e_monster_stats_by_cr.csv')
print(f"📊 Loaded {len(lazy_5e)} CR baselines from Lazy 5e")

print("\n🔍 Lazy 5e Baseline Stats Preview:")
display(lazy_5e.head(10))

Loaded 382 monsters
Columns: ['Name', 'Size', 'Type', 'Alignment', 'HP', 'AC', 'Speed', 'Challenge_Rating', 'XP', 'STR']...
📊 Loaded 324 monsters (2014 5e only, excluding 2024 additions)
📊 Loaded 34 CR baselines from Lazy 5e

🔍 Lazy 5e Baseline Stats Preview:


,CR,Eqv_Char_Lvl,AC_DC,HP,Attack_Bonus,Damage_Round,Num_Attacks,Damage_Attack,Example_Monster
0,0,< 1,10,3 (2-4),2,2,1,2 (1d4),"Commoner, rat, spider"
1,1/8,< 1,11,9 (7-11),3,3,1,4 (1d6 + 1),"Bandit, cultist, giant rat"
2,1/4,1,11,13 (10-16),3,5,1,5 (1d6 + 2),"Acolyte, skeleton, wolf"
3,1/2,2,12,22 (17-28),4,8,2,4 (1d4 + 2),"Black bear, scout, shadow"
4,1,3,12,33 (25-41),5,12,2,6 (1d8 + 2),"Dire wolf, specter, spy"
5,2,5,13,45 (34-56),5,17,2,9 (2d6 + 2),"Ghast, ogre, priest"
6,3,7,13,65 (49-81),5,23,2,12 (2d8 + 3),"Knight, mummy, werewolf"
7,4,9,14,84 (64-106),6,28,2,14 (3d8 + 1),"Ettin, ghost"
8,5,10,15,95 (71-119),7,35,3,12 (3d6 + 2),"Elemental, gladiator, vampire spawn"
9,6,11,15,112 (84-140),7,41,3,14 (3d6 + 4),"Mage, medusa, wyvern"


In [86]:
# === CONDITION DEFINITIONS (D&D 5e) ===
# Source: https://roll20.net/compendium/dnd5e/Conditions
# These definitions inform how we value condition-inflicting abilities

CONDITION_DEFINITIONS = {
    'blinded': {
        'description': "Can't see, auto-fails sight-based checks. Attacks against have advantage, its attacks have disadvantage.",
        'severity': 'high',  # Significant combat impact
    },
    'charmed': {
        'description': "Can't attack or harm the charmer. Charmer has advantage on social checks.",
        'severity': 'medium',  # Situational, doesn't affect all enemies
    },
    'deafened': {
        'description': "Can't hear, auto-fails hearing-based checks.",
        'severity': 'low',  # Minimal combat impact
    },
    'frightened': {
        'description': "Disadvantage on ability checks and attacks while source visible. Can't willingly approach source.",
        'severity': 'medium',  # Good debuff but conditional
    },
    'grappled': {
        'description': "Speed becomes 0, can't benefit from speed bonuses. Ends if grappler incapacitated.",
        'severity': 'medium',  # Locks down movement
    },
    'incapacitated': {
        'description': "Can't take actions or reactions.",
        'severity': 'high',  # Complete action denial
    },
    'paralyzed': {
        'description': "Incapacitated, can't move or speak. Auto-fails STR/DEX saves. Attacks have advantage, crits within 5ft.",
        'severity': 'very_high',  # Devastating - combines multiple effects
    },
    'petrified': {
        'description': "Turned to stone. Incapacitated, resistant to all damage, immune to poison/disease.",
        'severity': 'very_high',  # Effectively removes from combat
    },
    'poisoned': {
        'description': "Disadvantage on attack rolls and ability checks.",
        'severity': 'medium',  # Consistent debuff
    },
    'prone': {
        'description': "Must crawl to move. Attacks have disadvantage. Melee attacks against have advantage, ranged have disadvantage.",
        'severity': 'low',  # Easy to recover from (half movement)
    },
    'restrained': {
        'description': "Speed 0. Attacks against have advantage, its attacks have disadvantage. DEX saves have disadvantage.",
        'severity': 'high',  # Strong control + offensive debuff
    },
    'stunned': {
        'description': "Incapacitated, can't move, speaks falteringly. Auto-fails STR/DEX saves. Attacks against have advantage.",
        'severity': 'very_high',  # Near-paralysis
    },
    'unconscious': {
        'description': "Incapacitated, can't move/speak, unaware. Drops items, falls prone. Attacks are crits within 5ft.",
        'severity': 'very_high',  # Complete incapacitation
    },
}

# Severity rankings for reference:
# very_high: paralyzed, petrified, stunned, unconscious (target is essentially out of combat)
# high: blinded, incapacitated, restrained (major combat impairment)
# medium: charmed, frightened, grappled, poisoned (significant but conditional/recoverable)
# low: deafened, prone (minimal or easily recovered)

print("📋 Condition Definitions Loaded:")
for condition, info in CONDITION_DEFINITIONS.items():
    print(f"   {condition:15s} [{info['severity']:9s}]: {info['description'][:60]}...")


📋 Condition Definitions Loaded:
   blinded         [high     ]: Can't see, auto-fails sight-based checks. Attacks against ha...
   charmed         [medium   ]: Can't attack or harm the charmer. Charmer has advantage on s...
   deafened        [low      ]: Can't hear, auto-fails hearing-based checks....
   frightened      [medium   ]: Disadvantage on ability checks and attacks while source visi...
   grappled        [medium   ]: Speed becomes 0, can't benefit from speed bonuses. Ends if g...
   incapacitated   [high     ]: Can't take actions or reactions....
   paralyzed       [very_high]: Incapacitated, can't move or speak. Auto-fails STR/DEX saves...
   petrified       [very_high]: Turned to stone. Incapacitated, resistant to all damage, imm...
   poisoned        [medium   ]: Disadvantage on attack rolls and ability checks....
   prone           [low      ]: Must crawl to move. Attacks have disadvantage. Melee attacks...
   restrained      [high     ]: Speed 0. Attacks against have ad

# Actual Feature Engineering

## Parse Basic Features

### Baselines

In [87]:
# Convert Lazy 5e CR to numeric
lazy_5e['cr_numeric'] = lazy_5e['CR'].apply(parse_cr)


# Parse HP from Lazy 5e (extract average from "65 (49-81)" format)
lazy_5e['hp_baseline'] = lazy_5e['HP'].apply(parse_hp_avg)

# Adjust HP baselines: +50% for CR <= 1, +20% for CR >= 2
lazy_5e['hp_baseline'] = lazy_5e.apply(adjust_hp_baseline, axis=1)
print("✅ Adjusted HP baselines: +50% for CR ≤ 1, +20% for CR ≥ 2")

# Parse Attack Bonus - handles both numeric (3) and "+3" format
lazy_5e['attack_baseline'] = lazy_5e['Attack_Bonus'].apply(parse_bonus)
lazy_5e['ac_baseline'] = lazy_5e['AC_DC']
lazy_5e['dpr_baseline'] = lazy_5e['Damage_Round']

print("✅ Parsed Lazy 5e baselines")


# Add size_ordinal baseline (Medium for CR < 12, Large for CR >= 12)
lazy_5e['size_ordinal_baseline'] = lazy_5e['cr_numeric'].apply(
    lambda cr: 2 if cr < 12 else 3  # 2 = Medium, 3 = Large
)

# Add speed baselines (constant 30 for all CR)
lazy_5e['speed_ground_baseline'] = 30
lazy_5e['max_speed_baseline'] = 30

print("✅ Added size_ordinal, speed_ground, and max_speed baselines")

print("\n📊 Baseline Stats by CR:")
display(lazy_5e[['cr_numeric', 'hp_baseline', 'ac_baseline', 'attack_baseline', 'dpr_baseline']].head(15))

✅ Adjusted HP baselines: +50% for CR ≤ 1, +20% for CR ≥ 2
✅ Parsed Lazy 5e baselines
✅ Added size_ordinal, speed_ground, and max_speed baselines

📊 Baseline Stats by CR:


,cr_numeric,hp_baseline,ac_baseline,attack_baseline,dpr_baseline
0,0.000,4.5,10,2,2
1,0.125,13.5,11,3,3
2,0.250,19.5,11,3,5
3,0.500,33.0,12,4,8
4,1.000,49.5,12,5,12
5,2.000,54.0,13,5,17
6,3.000,78.0,13,5,23
7,4.000,100.8,14,6,28
8,5.000,114.0,15,7,35
9,6.000,134.4,15,7,41


In [88]:
# Create interpolation functions for baselines
# cr_values = lazy_5e['cr_numeric'].values
# hp_baseline_interp = interp1d(cr_values, lazy_5e['hp_baseline'].values, 
#                                kind='linear', bounds_error=False, fill_value='extrapolate')
# ac_baseline_interp = interp1d(cr_values, lazy_5e['ac_baseline'].values,
#                                kind='linear', bounds_error=False, fill_value='extrapolate')
# attack_baseline_interp = interp1d(cr_values, lazy_5e['attack_baseline'].values,
#                                    kind='linear', bounds_error=False, fill_value='extrapolate')
# dpr_baseline_interp = interp1d(cr_values, lazy_5e['dpr_baseline'].values,
#                                 kind='linear', bounds_error=False, fill_value='extrapolate')
# # DC baseline uses same values as AC (from AC_DC column)
# dc_baseline_interp = interp1d(cr_values, lazy_5e['ac_baseline'].values,
#                                kind='linear', bounds_error=False, fill_value='extrapolate')


# # Size ordinal baseline (step function at CR 12)
# size_ordinal_baseline_interp = interp1d(
#     cr_values,
#     lazy_5e['size_ordinal_baseline'].values,
#     kind='previous',  # Use step function instead of linear
#     bounds_error=False,
#     fill_value=(2, 3)  # Medium below range, Large above range
# )
# print("✅ Created baseline interpolation functions")

### Main df

#### Core - CR, AC, HP

In [89]:
# Parse CR
df['cr_numeric'] = df['Challenge_Rating'].apply(parse_cr)

# Parse HP
df['actual_hp'] = df['HP'].apply(parse_hp)

# Parse AC
df['ac_value'] = df['AC'].apply(parse_ac)

print(f"CR range: {df['cr_numeric'].min()} - {df['cr_numeric'].max()}")
print(f"HP range: {df['actual_hp'].min()} - {df['actual_hp'].max()}")

CR range: 0.0 - 30.0
HP range: 1 - 676


#### Speeds

In [90]:
# Parse speeds
df['speed_ground'] = df['Speed'].apply(lambda x: parse_speed(x, 'ground'))
df['speed_fly'] = df['Speed'].apply(lambda x: parse_speed(x, 'fly'))
df['speed_swim'] = df['Speed'].apply(lambda x: parse_speed(x, 'swim'))
df['speed_burrow'] = df['Speed'].apply(lambda x: parse_speed(x, 'burrow'))
df['speed_climb'] = df['Speed'].apply(lambda x: parse_speed(x, 'climb'))

df['max_speed'] = df[['speed_ground', 'speed_fly', 'speed_swim', 'speed_burrow', 'speed_climb']].max(axis=1)
df['movement_types_count'] = (df[['speed_ground', 'speed_fly', 'speed_swim', 'speed_burrow', 'speed_climb']] > 0).sum(axis=1)
df['has_flying'] = (df['speed_fly'] > 0).astype(int)


#### Size

In [91]:
# Parse size
df['size_ordinal'] = df['Size'].map(SIZE_ORDINAL_MAP).fillna(2)

#### Proficiencies

In [92]:
# Parse proficiencies
df['save_proficiency_count'] = df['Saving_Throws'].apply(count_proficiencies)
df['skill_proficiency_count'] = df['Skills'].apply(count_proficiencies)
df['resistance_count'] = df['Resistances'].apply(count_proficiencies)
df['immunity_count'] = df['Immunities'].apply(count_proficiencies)
df['vulnerability_count'] = df['Vulnerabilities'].apply(count_proficiencies)
df['condition_immunity_count'] = df['Condition_Immunities'].apply(count_proficiencies)

print("Proficiencies parsed")

Proficiencies parsed


#### Senses

In [93]:
# Parse senses
df['has_darkvision'] = df['Senses'].apply(lambda x: has_sense(x, 'darkvision'))
df['darkvision_range'] = df['Senses'].apply(lambda x: parse_sense_range(x, 'darkvision'))
df['has_blindsight'] = df['Senses'].apply(lambda x: has_sense(x, 'blindsight'))
df['has_truesight'] = df['Senses'].apply(lambda x: has_sense(x, 'truesight'))
df['has_tremorsense'] = df['Senses'].apply(lambda x: has_sense(x, 'tremorsense'))
df['passive_perception'] = df['Senses'].apply(parse_passive_perception)

print("Senses parsed")

Senses parsed


#### Abilities and Legendary Count

In [94]:
# Parse ability counts
df['trait_count'] = df['Traits'].apply(count_abilities)
df['action_count'] = df['Actions'].apply(count_abilities)
df['reaction_count'] = df['Reactions'].apply(count_abilities)
df['bonus_action_count'] = df['Bonus_Actions'].apply(count_abilities) if 'Bonus_Actions' in df.columns else 0

# Legendary actions
df[['has_legendary_actions', 'legendary_action_count', 'legendary_actions_per_round']] = df['Legendary_Actions'].apply(
    lambda x: pd.Series(parse_legendary_actions(x))
)

df['total_ability_count'] = df['trait_count'] + df['action_count'] + df['reaction_count'] + df['legendary_action_count']

print("Ability counts parsed")

Ability counts parsed


## Parse Combat Stats

### Attack Bonus

In [95]:
# Parse attack bonus
df['highest_attack_bonus'] = df['Actions'].apply(parse_attack_bonus)

### Save DC and Overrides

In [96]:

# Parse save DC
combined_abilities = (df['Traits'].fillna('') + ' ' + df['Actions'].fillna('') + ' ' +
                     df['Reactions'].fillna('') + ' ' + df['Legendary_Actions'].fillna(''))
df['highest_save_dc'] = combined_abilities.apply(parse_save_dc)

# DC Overrides (creatures with thematically high DCs)
DC_OVERRIDES = {
    'Green Hag': 14,  # DC 20 is for Illusory Appearance (thematic, not combat)
}
for creature, dc in DC_OVERRIDES.items():
    df.loc[df['Name'] == creature, 'highest_save_dc'] = dc

print(f"Applied {len(DC_OVERRIDES)} DC overrides")

Applied 1 DC overrides


### DPRs

In [97]:
# Parse DPR
df['estimated_dpr'] = df['Actions'].apply(parse_dpr_from_json)

# Add charge/pounce bonus DPR
df['charge_bonus_dpr'] = df.apply(lambda row: parse_charge_bonus_attack(row['Traits'], row['Actions']), axis=1)
df['estimated_dpr'] = df['estimated_dpr'] + df['charge_bonus_dpr']

# Parse legendary DPR
df['legendary_dpr'] = df['Legendary_Actions'].apply(parse_legendary_actions_dpr)
df['total_dpr'] = df['estimated_dpr'] + df['legendary_dpr']

print(f"DPR range: {df['total_dpr'].min():.1f} - {df['total_dpr'].max():.1f}")

DPR range: 0.0 - 148.0


## Parse Special Traits

### Special traits from combined abilities

In [98]:
# Special traits from combined abilities
df['has_legendary_resistance'] = combined_abilities.str.contains('legendary resistance', case=False, na=False).astype(int)
df['has_magic_resistance'] = combined_abilities.str.contains('magic resistance', case=False, na=False).astype(int)
df['has_regeneration'] = combined_abilities.str.contains('regeneration', case=False, na=False).astype(int)
df['has_spellcasting'] = combined_abilities.str.contains('spellcasting', case=False, na=False).astype(int)
df['spellcaster_level'] = combined_abilities.apply(extract_spellcaster_level)
df['has_grapple'] = combined_abilities.str.contains('grapple|grappled', case=False, na=False).astype(int)

print("Special traits parsed")

Special traits parsed


### Inflicts Conditions

In [99]:
# Parse legendary conditions
df['legendary_conditions'] = df['Legendary_Actions'].apply(parse_legendary_conditions)

In [100]:
# Condition infliction features
for condition in CONDITIONS:
    feature_name = f'inflicts_{condition}'
    df[feature_name] = combined_abilities.str.contains(condition, case=False, na=False).astype(int)
    # Also check legendary actions
    df[feature_name] = df.apply(
        lambda row: 1 if (row[feature_name] == 1 or condition in row['legendary_conditions']) else 0,
        axis=1
    )

# Inflicts prone (Phase 2 feature)
df['inflicts_prone'] = combined_abilities.str.contains('prone', case=False, na=False).astype(int)
df['inflicts_prone'] = df.apply(
    lambda row: 1 if (row['inflicts_prone'] == 1 or 'prone' in row['legendary_conditions']) else 0,
    axis=1
)

print(f"{len(CONDITIONS)} condition features added")
print(f"Creatures that inflict prone: {df['inflicts_prone'].sum()} ({df['inflicts_prone'].mean()*100:.1f}%)")

10 condition features added
Creatures that inflict prone: 64 (19.8%)


### Advantage/disadvantage conditions

In [101]:
# Advantage/disadvantage conditions
df['has_advantage_condition'] = df.apply(has_advantage_condition, axis=1)
df['has_disadvantage_condition'] = df.apply(has_disadvantage_condition, axis=1)
df['has_attackers_advantage'] = df.apply(has_attackers_advantage, axis=1)

print(f"Advantage conditions: {df['has_advantage_condition'].sum()} ({df['has_advantage_condition'].mean()*100:.1f}%)")
print(f"Disadvantage conditions: {df['has_disadvantage_condition'].sum()} ({df['has_disadvantage_condition'].mean()*100:.1f}%)")

Advantage conditions: 32 (9.9%)
Disadvantage conditions: 9 (2.8%)


## Calculate Baselines and Deviations

### Calculate Baselines

#### Core Attributes

In [102]:
# Calculate baselines
df['hp_baseline'] = df['cr_numeric'].apply(get_baseline_hp)
df['ac_baseline'] = df['cr_numeric'].apply(get_baseline_ac)
df['attack_baseline'] = df['cr_numeric'].apply(get_baseline_attack)
df['dpr_baseline'] = df['cr_numeric'].apply(get_baseline_dpr)
df['dc_baseline'] = df['cr_numeric'].apply(get_baseline_dc)
df['size_ordinal_baseline'] = df['cr_numeric'].apply(get_baseline_size_ordinal)

df['speed_ground_baseline'] = df['cr_numeric'].apply(get_baseline_speed_ground)

#### Flying

In [103]:
# Calculate fly speed baseline (only for flyers)
df['fly_speed_baseline'] = df['cr_numeric'].apply(get_fly_speed_baseline)
df['speed_fly_deviation'] = df.apply(
    lambda row: row['speed_fly'] - row['fly_speed_baseline'] if row['speed_fly'] > 0 else 0,
    axis=1
)

#### Darkvision

In [104]:

# Calculate darkvision baseline (only for creatures with darkvision)
df['darkvision_baseline'] = df['cr_numeric'].apply(get_darkvision_baseline)
df['darkvision_deviation'] = df.apply(
    lambda row: row['darkvision_range'] - row['darkvision_baseline'] if row['darkvision_range'] > 0 else 0,
    axis=1
)

In [105]:
print("Baselines calculated")

Baselines calculated


### Calculate Deviations

In [106]:
# Calculate deviations
df['ac_deviation'] = df['ac_value'] - df['ac_baseline']
df['attack_deviation'] = df['highest_attack_bonus'] - df['attack_baseline']
df['dpr_deviation'] = df['total_dpr'] - df['dpr_baseline']

# Save DC deviation: only calculate if creature has a save DC (otherwise 0, no penalty)
df['save_dc_deviation'] = df.apply(
    lambda row: row['highest_save_dc'] - row['dc_baseline'] if row['highest_save_dc'] > 0 else 0,
    axis=1
)

# Add passive perception bonus to save_dc_deviation if it exceeds baseline + 1
df['passive_perception_bonus'] = (df['passive_perception'] - (df['dc_baseline'] + 1)).clip(lower=0)
df['save_dc_deviation'] = df['save_dc_deviation'] + df['passive_perception_bonus']

df['size_ordinal_deviation'] = df['size_ordinal'] - df['size_ordinal_baseline']
df['speed_ground_deviation'] = df['speed_ground'] - df['speed_ground_baseline']

print("Deviations calculated")

Deviations calculated


## Phase 1.5: Resistance/Immunity Penalties

In [107]:
# Filter to valid HP
df_valid = df[df['actual_hp'] > 0].copy()
print(f"Valid samples: {len(df_valid)} monsters with HP > 0")

# Phase 1: Baseline HP
df_valid['hp_after_phase1'] = df_valid['hp_baseline']

# Calculate resistance/immunity penalties
df_valid['resistance_multiplier'] = df_valid['cr_numeric'].apply(get_resistance_multiplier)
df_valid['immunity_multiplier'] = df_valid['cr_numeric'].apply(get_immunity_multiplier)

df_valid['resistance_penalty'] = (
    df_valid['resistance_multiplier'] * 
    df_valid['hp_after_phase1'] * 
    (df_valid['resistance_count'] > 0)
)
df_valid['immunity_penalty'] = (
    df_valid['immunity_multiplier'] * 
    df_valid['hp_after_phase1'] * 
    (df_valid['immunity_count'] > 0)
)

df_valid['total_defensive_penalty'] = df_valid['immunity_penalty'] + df_valid['resistance_penalty']

# Apply 75% cap
df_valid['total_defensive_penalty'] = df_valid['total_defensive_penalty'].clip(
    upper=0.75 * df_valid['hp_after_phase1']
)

df_valid['hp_after_phase1_5'] = df_valid['hp_after_phase1'] - df_valid['total_defensive_penalty']

print("Phase 1.5 complete")

Valid samples: 324 monsters with HP > 0
Phase 1.5 complete


## Split by CR and Apply Phase 2 Penalties

In [108]:
# Split by CR tier
df_cr1 = df_valid[df_valid['cr_numeric'] < 1.0].copy()
df_cr2 = df_valid[(df_valid['cr_numeric'] >= 1.0) & (df_valid['cr_numeric'] <= 4.0)].copy()
df_cr3 = df_valid[(df_valid['cr_numeric'] >= 5.0) & (df_valid['cr_numeric'] <= 10.0)].copy()
df_cr4 = df_valid[(df_valid['cr_numeric'] >= 11.0) & (df_valid['cr_numeric'] <= 16.0)].copy()
df_cr5 = df_valid[df_valid['cr_numeric'] > 16.0].copy()

print(f"CR < 1:    {len(df_cr1)} monsters")
print(f"CR 1-4:    {len(df_cr2)} monsters")
print(f"CR 5-10:   {len(df_cr3)} monsters")
print(f"CR 11-16:  {len(df_cr4)} monsters")
print(f"CR > 16:   {len(df_cr5)} monsters")

CR < 1:    113 monsters
CR 1-4:    99 monsters
CR 5-10:   65 monsters
CR 11-16:  27 monsters
CR > 16:   20 monsters


In [109]:
def apply_phase2_penalties(df_tier, tier_key):
    """Apply Phase 2 penalties to a CR tier dataframe.
    
    Gets to "residual HP
    """
    penalties = PHASE2_PENALTIES[tier_key]
    
    df_tier['hp_after_phase2'] = df_tier['hp_after_phase1_5'].copy()
    for feature, penalty in penalties.items():
        if feature in df_tier.columns:
            df_tier['hp_after_phase2'] += df_tier[feature] * penalty
    
    # Calculate scaled features
    df_tier['has_legendary_resistance_scaled'] = df_tier['has_legendary_resistance'] * df_tier['hp_after_phase2']
    df_tier['has_magic_resistance_scaled'] = df_tier['has_magic_resistance'] * df_tier['hp_after_phase2']
    df_tier['has_regeneration_scaled'] = df_tier['has_regeneration'] * df_tier['hp_after_phase2']
    
    # Calculate residual HP
    df_tier['residual_hp'] = df_tier['actual_hp'] - df_tier['hp_after_phase2']
    
    return df_tier

# Apply Phase 2 to each tier
df_cr1 = apply_phase2_penalties(df_cr1, 'cr1')
df_cr2 = apply_phase2_penalties(df_cr2, 'cr2')
df_cr3 = apply_phase2_penalties(df_cr3, 'cr3')
df_cr4 = apply_phase2_penalties(df_cr4, 'cr4')
df_cr5 = apply_phase2_penalties(df_cr5, 'cr5')

print("Phase 2 penalties applied to all tiers")

Phase 2 penalties applied to all tiers


In [110]:
# Add family and cr_tier for train/test splitting
for df_tier, tier_key in [(df_cr1, 'cr1'), (df_cr2, 'cr2'), (df_cr3, 'cr3'), (df_cr4, 'cr4'), (df_cr5, 'cr5')]:
    df_tier['family'] = df_tier['Name'].apply(extract_family)
    df_tier['cr_tier'] = tier_key

In [111]:
df_cr1[['Name', 'hp_after_phase2', 'residual_hp']]

,Name,hp_after_phase2,residual_hp
1,Acolyte,21.375,-12.375
34,Ape,27.250,-8.250
39,Awakened Shrub,5.750,4.250
41,Axe Beak,16.375,2.625
44,Baboon,0.875,2.125
...,...,...,...
352,Warhorse Skeleton,17.000,5.000
356,Weasel,-5.250,6.250
366,Wolf,10.750,0.250
367,Worg,23.750,2.250


# Combine and Save

In [112]:
# Combine all tiers back together
df_engineered = pd.concat([df_cr1, df_cr2, df_cr3, df_cr4, df_cr5], ignore_index=True)
df_engineered = df_engineered.sort_values('cr_numeric')

print(f"Total engineered samples: {len(df_engineered)}")

Total engineered samples: 324


In [117]:
df_cr2['residual_hp'].sum()

np.float64(-896.3125)

In [114]:

# Select columns to export
columns_to_export = [
    # Identifying information
    'Name', 'Type', 'Size', 'Challenge_Rating', 'cr_numeric',
    
    # Original stats
    'HP', 'actual_hp', 'AC', 'ac_value',
    
    # Predictions and errors
    'predicted_hp', 'hp_delta', 'hp_delta_pct', 'hp_delta_pct_percentile',
    
    # Baselines
    'hp_baseline', 'ac_baseline', 'attack_baseline', 'dpr_baseline', 'dc_baseline',
    
    # Combat stats
    'highest_attack_bonus', 'highest_save_dc', 'estimated_dpr', 'legendary_dpr', 'total_dpr',
    
    # Phase 2: Deviations
    'ac_deviation', 'attack_deviation', 'dpr_deviation', 'save_dc_deviation',
    
    # Phase intermediates
    'hp_after_phase1_5', 'hp_after_phase2', 'residual_hp',
    
    # Phase 3: Scaled features (if they exist)
    'has_legendary_resistance_scaled', 
    'has_magic_resistance_scaled', 'has_regeneration_scaled',
    
    # Phase 3: Movement
    'speed_ground', 'speed_fly', 'speed_swim', 'speed_burrow', 'speed_climb',
    'max_speed', 'movement_types_count', 'has_flying',
    
    # Phase 3: Defenses
    'save_proficiency_count', 'skill_proficiency_count',
    'resistance_count', 'immunity_count', 'vulnerability_count', 
    'condition_immunity_count',
    
    # Phase 3: Senses
    'has_darkvision', 'darkvision_deviation', 'has_blindsight', 
    'has_truesight', 'has_tremorsense', 'passive_perception',
    
    # Phase 3: Abilities
    'trait_count', 'action_count', 'reaction_count', 'bonus_action_count',
    'legendary_action_count', 'legendary_actions_per_round',
    'total_ability_count',
    
    # Phase 3: Special abilities
    'has_legendary_actions', 'has_legendary_resistance', 
    'has_magic_resistance', 'has_regeneration',
    'has_spellcasting', 'spellcaster_level',
    'size_ordinal', 'has_grapple',
    
    # Phase 3: Condition infliction
    'inflicts_poisoned', 'inflicts_blinded', 'inflicts_charmed',
    'inflicts_deafened', 'inflicts_frightened', 'inflicts_incapacitated',
    'inflicts_paralyzed', 'inflicts_petrified', 'inflicts_prone',
    'inflicts_restrained', 'inflicts_stunned'
]
print(len(columns_to_export))

79


In [115]:
# Save to parquet
output_path = IO_DIR + '/engineered_features.parquet'
df_engineered.to_parquet(output_path, index=False)

print(f"Saved {len(df_engineered)} monsters with {len(df_engineered.columns)} features")
print(f"Output: {output_path}")

Saved 324 monsters with 130 features
Output: ./notebooks/notebooks_io/engineered_features.parquet


In [116]:
# Summary of key features
phase3_features = get_phase3_features()
print(f"\nPhase 2 features: {len(PHASE2_FEATURES)}")
print(f"Phase 3 features: {len(phase3_features)}")
print(f"\nSample data:")
df_engineered[['Name', 'cr_numeric', 'actual_hp', 'hp_baseline', 'hp_after_phase1_5', 'hp_after_phase2', 'residual_hp']].head(10)


Phase 2 features: 9
Phase 3 features: 38

Sample data:


,Name,cr_numeric,actual_hp,hp_baseline,hp_after_phase1_5,hp_after_phase2,residual_hp
2,Awakened Shrub,0.0,10,4.5,3.375,5.750,4.250
22,Deer,0.0,4,4.5,4.500,-0.375,4.375
28,Eagle,0.0,3,4.5,4.500,-6.875,9.875
32,Frog,0.0,1,4.5,4.500,8.500,-7.500
37,Giant Fire Beetle,0.0,4,4.5,4.500,0.125,3.875
78,Rat,0.0,1,4.5,4.500,9.250,-8.250
79,Raven,0.0,1,4.5,4.500,-4.250,5.250
72,Owl,0.0,1,4.5,4.500,-0.750,1.750
70,Octopus,0.0,3,4.5,4.500,-1.750,4.750
56,Homunculus,0.0,5,4.5,3.375,-6.875,11.875
